In [1]:
import pandas as pd
import numpy as np
import boto3
import os
import json
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample

In [2]:
file_paths = {
    'spfile': 'ICPSR_31421/DS0001/31421-0001-Data.dta',   
    'capi':   'ICPSR_31421/DS0002/31421-0002-Data.dta',   
    'acasi':  'ICPSR_31421/DS0003/31421-0003-Data.dta',   
    'cidi':   'ICPSR_31421/DS0004/31421-0004-Data.dta',  
    'exam':   'ICPSR_31421/DS0005/31421-0005-Data.dta', 
    'lab':    'ICPSR_31421/DS0006/31421-0006-Data.dta',}

dfs = {}
for name, path in file_paths.items():
    dfs[name] = pd.read_stata(path, convert_categoricals=False)
    print(f"{name:8s}: {dfs[name].shape[0]} rows, {dfs[name].shape[1]} cols")

# merge all files on SP_ID (keep the first occurrence of any column name that appears in more than one file)
df_all = dfs['spfile'].copy()
for name in ['capi', 'acasi', 'cidi', 'exam', 'lab']:
    df = dfs[name]
    new_cols = ['SP_ID'] + [c for c in df.columns if c not in df_all.columns]
    df_all = pd.merge(df_all, df[new_cols], on='SP_ID', how='outer')

print(f"\nMerged dataset: {df_all.shape[0]} rows, {df_all.shape[1]} columns")

spfile  : 1999 rows, 44 cols
capi    : 1999 rows, 395 cols
acasi   : 1725 rows, 75 cols
cidi    : 1817 rows, 288 cols
exam    : 1999 rows, 43 cols
lab     : 1853 rows, 29 cols

Merged dataset: 1999 rows, 827 columns


In [7]:
# variables with near-complete data for HCV+
required_vars = [
    'RIAGENDR',   # Gender
    'RIAAGEYR',   # Age
    'RACE_ETH',   # Race/ethnicity
    'DMQ140',     # Diabetes
    'HIQ012',     # Health insurance coverage
    'HUQ010',     # General health status
    'BMXBMI',     # BMI
    'BMXWAIST',   # Waist circumference
    'BPXSAR',     # Avg systolic BP
    'BPXDAR',     # Avg diastolic BP
    'LBXGH',      # HbA1c
    'LBXGLU',     # Glucose
    'LBXTC',      # Total cholesterol
    'LBXHDD',     # HDL
    'LBXBPB',     # Lead
    'LBXBCD',     # Cadmium
    'LBXTHG',     # Mercury
    'FSTHOURS',   # Fasting hours
]

df_complete = df_all.dropna(subset=required_vars).copy()

# allow up to 1 missing value per variable
RANDOM_STATE = 42
SAMPLE_PER_GROUP = 35

hcv_pos_all = df_all[df_all['HCV'] == 1].copy()
hcv_neg_all = df_all[df_all['HCV'] == 2].copy()

# check per-variable missingness within the full HCV+ pool (n=35)
pos_missing_counts = hcv_pos_all[required_vars].isna().sum()
print("Missing counts per variable among ALL 35 HCV+ individuals:")
print(pos_missing_counts[pos_missing_counts > 0])

# use all 35 HCV+ individuals (tolerating the single missing cell)
sample_pos = hcv_pos_all.sample(n=min(SAMPLE_PER_GROUP, len(hcv_pos_all)), random_state=RANDOM_STATE)

# HCV- draws 35 fully complete)
hcv_neg_pool = hcv_neg_all.dropna(subset=required_vars)
sample_neg = hcv_neg_pool.sample(n=min(SAMPLE_PER_GROUP, len(hcv_neg_pool)), random_state=RANDOM_STATE)

sample_df = (pd.concat([sample_pos, sample_neg]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True))

print(f"\nFinal sample size: {len(sample_df)} ({len(sample_pos)} HCV+, {len(sample_neg)} HCV-)")

Missing counts per variable among ALL 35 HCV+ individuals:
HIQ012    1
HUQ010    1
dtype: int64

Final sample size: 70 (35 HCV+, 35 HCV-)


In [10]:
gender_map = {
    1: "Male",
    2: "Female"}

race_eth_map = {
    1: "Non-Hispanic White",
    2: "Non-Hispanic Black",
    3: "Non-Hispanic Asian",
    4: "Hispanic",
    5: "Non-Hispanic Other/Multi-racial"}

education_map = {
    0: "never attended school or kindergarten only",
    1: "completed 1st grade",
    2: "completed 2nd grade",
    3: "completed 3rd grade",
    4: "completed 4th grade",
    5: "completed 5th grade",
    6: "completed 6th grade",
    7: "completed 7th grade",
    8: "completed 8th grade",
    9: "completed 9th grade",
    10: "completed 10th grade",
    11: "completed 11th grade",
    12: "completed 12th grade, no diploma",
    13: "high school graduate",
    14: "GED or equivalent",
    15: "some college, no degree",
    16: "associate degree (occupational/technical/vocational)",
    17: "associate degree (academic program)",
    18: "bachelor's degree",
    19: "master's degree",
    20: "professional school degree (e.g., MD, JD)",
    21: "doctoral degree (e.g., PhD, EdD)",
    99: "unknown education level"}

insurance_map = {
    1: "has health insurance coverage",
    2: "does not have health insurance coverage",
    9: "unknown insurance status"}

health_status_map = {
    1: "excellent",
    2: "very good",
    3: "good",
    4: "fair",
    5: "poor"}

smoking_map = {
    1: "smokes cigarettes every day",
    2: "smokes cigarettes some days",
    3: "does not currently smoke cigarettes"}

yes_no_map = {
    1: "yes",
    2: "no",
    7: "refused to answer",
    9: "does not know"}

def safe_label(value, mapping, default="Not reported"):
    if pd.isna(value):
        return default
    return mapping.get(value, default)

def safe_num(value, fmt="{:.1f}"):
    if pd.isna(value):
        return "Not reported"
    return fmt.format(value)

# create prompt variables
prompt_list = []

for idx, row in sample_df.iterrows():
    age = safe_num(row["RIAAGEYR"], "{:.0f}")
    sex = safe_label(row["RIAGENDR"], gender_map, "unknown sex")
    race = safe_label(
        row["RACE_ETH"],
        race_eth_map,
        "unspecified race/ethnicity")
    education = safe_label(
        row["DMQ140"],
        education_map,
        "unknown education level")
    insurance = safe_label(row["HIQ012"], insurance_map)
    health = safe_label(row["HUQ010"], health_status_map)
    smoking = safe_label(
        row["SMQ040"],
        smoking_map,
        "unknown smoking status")
    street_drugs = safe_label(
        row["DUQ100"],
        yes_no_map,
        "unknown")
    needle_use = safe_label(
        row["DUQ120"],
        yes_no_map,
        "unknown")

    bmi = safe_num(row["BMXBMI"], "{:.1f}")
    waist = safe_num(row["BMXWAIST"], "{:.1f}")
    sbp = safe_num(row["BPXSAR"], "{:.0f}")
    dbp = safe_num(row["BPXDAR"], "{:.0f}")
    hba1c = safe_num(row["LBXGH"], "{:.1f}")
    cholesterol = safe_num(row["LBXTC"], "{:.0f}")
    hdl = safe_num(row["LBXHDD"], "{:.0f}")

    prompt = (
        "This individual was a resident of New York City in 2004. "
        "The following individual characteristics were collected during that period.\n\n"

        "Individual profile:\n"
        f"- Age: {age} years\n"
        f"- Sex: {sex}\n"
        f"- Race/ethnicity: {race}\n"
        f"- Education: {education}\n"
        f"- Health insurance status: {insurance}\n"
        f"- Self-reported general health status: {health}\n"
        f"- Smoking status: {smoking}\n"
        f"- Ever used cocaine or other street drugs (not marijuana): "
        f"{street_drugs}\n"
        f"- Ever used a needle to take street drugs: "
        f"{needle_use}\n"
        f"- Body mass index (BMI): {bmi} kg/m²\n"
        f"- Waist circumference: {waist} cm\n"
        f"- Average systolic blood pressure: {sbp} mmHg\n"
        f"- Average diastolic blood pressure: {dbp} mmHg\n"
        f"- Hemoglobin A1c: {hba1c}%\n"
        f"- Total cholesterol: {cholesterol} mg/dL\n"
        f"- HDL cholesterol: {hdl} mg/dL\n\n"

        "Question:\n"
        "Estimate the individual's relative likelihood of HCV infection based on the "
        "provided characteristics.\n\n"

        "Output Format:\n"
        "Return ONLY a single-line JSON object. Do not include explanations, "
        "markdown, code blocks, or additional text.\n\n"

        "The JSON must exactly match the following format:\n"
        "{\n"
        '  "HCV_risk_score": integer,\n'
        '  "risk_factor_1": "variable_name",\n'
        '  "risk_factor_2": "variable_name",\n'
        '  "risk_factor_3": "variable_name"\n'
        "}\n\n"

        "Mapping:\n"
        "- HCV_risk_score: integer from 0 to 100, where 0 indicates the lowest "
        "expected likelihood of HCV infection and 100 indicates the highest "
        "expected likelihood of HCV infection.\n"
        "- risk_factor_1, risk_factor_2, risk_factor_3: the three characteristics "
        "from the provided profile that most influenced the estimated HCV risk. "
        "Return only one of the following variable names: "
        "age, sex, race_ethnicity, education, health_insurance, "
        "self_rated_health, smoking_status, street_drug_use, "
        "injection_drug_use, BMI, waist_circumference, blood_pressure, "
        "HbA1c, cholesterol, HDL. "
        "If fewer than three factors are relevant, return \"None\" for the remaining fields."
    )

    prompt_list.append({
        "SP_ID": row["SP_ID"],
        "true_HCV": row["HCV"],
        "prompt": prompt})

print(prompt_list[0]["prompt"])

This individual was a resident of New York City in 2004. The following individual characteristics were collected during that period.

Individual profile:
- Age: 51 years
- Sex: Female
- Race/ethnicity: Non-Hispanic White
- Education: bachelor's degree
- Health insurance status: has health insurance coverage
- Self-reported general health status: excellent
- Smoking status: unknown smoking status
- Ever used cocaine or other street drugs (not marijuana): yes
- Ever used a needle to take street drugs: no
- Body mass index (BMI): 22.6 kg/m²
- Waist circumference: 83.3 cm
- Average systolic blood pressure: 118 mmHg
- Average diastolic blood pressure: 73 mmHg
- Hemoglobin A1c: 4.9%
- Total cholesterol: 201 mg/dL
- HDL cholesterol: 101 mg/dL

Question:
Estimate the individual's relative likelihood of HCV infection based on the provided characteristics.

Output Format:
Return ONLY a single-line JSON object. Do not include explanations, markdown, code blocks, or additional text.

The JSON must

In [ ]:
client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1")

model_id = "deepseek.v3.2"
responses_list = []

for prompt_entry in prompt_list:
    current_id = prompt_entry["SP_ID"]
    api_text = prompt_entry["prompt"]
    # retrieve the same person used to generate this prompt
    person_row = sample_df[sample_df["SP_ID"] == current_id]

    if person_row.empty:
        continue

    row = person_row.iloc[0]
    messages = [{
        "role": "user",
        "content": [{"text": api_text}]}]

    response = client.converse(
        modelId=model_id,
        messages=messages,
        inferenceConfig={
            "temperature": 0})

    raw_message_text = ""

    try:
        content_blocks = (
            response
            .get("output", {})
            .get("message", {})
            .get("content", []))
        texts = []
        for block in content_blocks:
            if isinstance(block, dict) and "text" in block:
                texts.append(block["text"])
        raw_message_text = " ".join(texts).strip()
    except Exception as e:
        print("Parsing error:", e)
        raw_message_text = ""

    try:
        structured_response = json.loads(raw_message_text)
    except json.JSONDecodeError:
        structured_response = raw_message_text

    entry = {
        # identifiers
        "SP_ID": current_id,
        "Model": model_id,
        "true_HCV": row["HCV"],

        # prompt variables
        "Age": safe_num(row["AGE"], "{:.0f}"),
        "Sex": row["sex_label"],
        "Race_ethnicity": row["race_group"],
        "Education": row["education_label"],
        "Health_insurance": row["insurance_label"],
        "Self_rated_health": row["health_label"],
        "Smoking_status": row["smoking_label"],
        "Street_drug_use": row["drug_use_label"],
        "Injection_drug_use": row["needle_use_label"],

        # clinical measurements
        "BMI": row["bmi"],
        "Waist_circumference": row["waist"],
        "Systolic_BP": row["sbp"],
        "Diastolic_BP": row["dbp"],
        "HbA1c": row["hba1c"],
        "Total_cholesterol": row["cholesterol"],
        "HDL": row["hdl"],

        # LLM output
        "response": structured_response
    }


    with open("outputs_nyc_hanes.jsonl", "a") as f:
        f.write(json.dumps(entry) + "\n")
    responses_list.append(entry)

In [ ]:
flat_data = []
with open("outputs_nyc_hanes.jsonl", "r") as f:
    for line in f:
        row = json.loads(line)  
        response = row.pop("response", "{}")  
        if isinstance(response, str):
            try:
                response = json.loads(response)  
            except json.JSONDecodeEarror:
                response = {}  
        flat_data.append({**row, **response}) 
pd.DataFrame(flat_data).to_csv("outputs.csv", index=False)

In [3]:
df = pd.read_csv("outputs_nyc_hanes.csv")

# 1 = positive, 2 = negative
df["HCV_binary"] = (df["true_HCV"] == 1).astype(int)

def bootstrap_auc(y_true, scores, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        y_sample = y_true.iloc[idx]
        s_sample = scores.iloc[idx]
        if len(y_sample.unique()) < 2: # must contain both classes
            continue
        aucs.append(roc_auc_score(y_sample, s_sample))
    return (np.mean(aucs), np.percentile(aucs, 2.5), np.percentile(aucs, 97.5))

# model performance
performance = []
for model, group in df.groupby("Model"):
    auc, lower, upper = bootstrap_auc(
        group["HCV_binary"],
        group["HCV_risk_score"])
    performance.append({
        "Model": model,
        "N": len(group),
        "AUC": round(auc, 3),
        "AUC_lower_95": round(lower, 3),
        "AUC_upper_95": round(upper, 3),
        "Mean_risk_HCV_positive":
            round(group.loc[group["HCV_binary"] == 1, "HCV_risk_score"].mean(), 2),
        "Mean_risk_HCV_negative":
            round(group.loc[group["HCV_binary"] == 0, "HCV_risk_score"].mean(), 2),
        "Median_risk_HCV_positive":
            round(group.loc[group["HCV_binary"] == 1, "HCV_risk_score"].median(), 2),
        "Median_risk_HCV_negative":
            round(group.loc[group["HCV_binary"] == 0, "HCV_risk_score"].median(), 2)})

performance_df = pd.DataFrame(performance)
print("\nMODEL PERFORMANCE\n")
print(performance_df.to_string(index=False))

# risk factor frequency
risk_factors = []
for model, group in df.groupby("Model"):
    factors = pd.concat([group["risk_factor_1"], group["risk_factor_2"], group["risk_factor_3"]])
    factors = (factors[factors != "None"].value_counts().reset_index())
    factors.columns = ["Risk_factor", "Count"]
    factors["Model"] = model
    risk_factors.append(factors)

risk_factor_df = pd.concat(risk_factors, ignore_index=True)

print("\nRISK FACTOR FREQUENCY\n")
print(risk_factor_df[["Model", "Risk_factor", "Count"]].sort_values([
    "Model", "Count"], ascending=[True, False]).to_string(index=False))

# risk score distributions by HCV status
risk_distribution = (df.groupby(["Model", "true_HCV"])["HCV_risk_score"]
    .agg(Mean="mean", Median="median", SD="std", Min="min", Max="max", N="count").reset_index())

risk_distribution["HCV_status"] = (risk_distribution["true_HCV"] .map({
        1: "Positive",  2: "Negative"}))

print("\nRISK SCORE DISTRIBUTION\n")
print(risk_distribution[
        ["Model", "HCV_status", "Mean", "Median", "SD", "Min", "Max", "N"]].to_string(index=False))


MODEL PERFORMANCE

          Model  N   AUC  AUC_lower_95  AUC_upper_95  Mean_risk_HCV_positive  Mean_risk_HCV_negative  Median_risk_HCV_positive  Median_risk_HCV_negative
claudesonnet4.6 70 0.854         0.755         0.929                   48.03                   17.14                      42.0                      18.0
  deepseek.v3.2 70 0.825         0.724         0.908                   38.66                    7.49                      30.0                       4.0
   gpt-oss-120b 70 0.783         0.661         0.884                   47.63                   23.06                      45.0                      22.0

RISK FACTOR FREQUENCY

          Model        Risk_factor  Count
claudesonnet4.6                age     59
claudesonnet4.6 injection_drug_use     36
claudesonnet4.6    street_drug_use     34
claudesonnet4.6     race_ethnicity     26
claudesonnet4.6          education     23
claudesonnet4.6   health_insurance      8
claudesonnet4.6                HDL      7
claudeso

In [20]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

df = pd.read_csv("outputs_nyc_hanes.csv")
df["HCV_binary"] = (df["true_HCV"] == 1).astype(int)

continuous = ["Age","BMI","Waist_circumference","Systolic_BP","Diastolic_BP","HbA1c","Total_cholesterol","HDL"]

binary = {
    "Sex": df["Sex"].astype(str).str.strip().str.lower().map({"female":1,"male":0}),
    "Smoking_status": df["Smoking_status"].astype(str).str.strip().str.lower().map({
        "smokes cigarettes every day":1,"smokes cigarettes some days":1,
        "does not currently smoke cigarettes":0
    }),
    "Street_drug_use": df["Street_drug_use"].astype(str).str.strip().str.lower().map({"yes":1,"no":0}),
    "Injection_drug_use": df["Injection_drug_use"].astype(str).str.strip().str.lower().map({"yes":1,"no":0})
}

def rho(x,y):
    tmp = pd.DataFrame({"x":x,"y":y}).replace([np.inf,-np.inf],np.nan).dropna()
    if len(tmp)<3 or tmp["x"].nunique()<2: return np.nan
    return spearmanr(tmp["x"],tmp["y"]).statistic

empirical = {v:rho(df[v],df["HCV_binary"]) for v in continuous}
empirical.update({v:rho(x,df["HCV_binary"]) for v,x in binary.items()})

results = []
for model,g in df.groupby("Model"):
    for v in continuous + list(binary):
        x = g[v] if v in continuous else binary[v].loc[g.index]
        emp_rho = empirical[v]
        llm_rho = rho(x,g["HCV_risk_score"])
        results.append({
            "Model":model,
            "Characteristic":v,
            "Empirical_rho":emp_rho,
            "LLM_risk_score_rho":llm_rho,
            "Absolute_difference":abs(emp_rho-llm_rho)
        })

results_df = pd.DataFrame(results)

summary = results_df.groupby("Model").agg(
    N_characteristics=("Absolute_difference","count"),
    Mean_absolute_difference=("Absolute_difference","mean")
).reset_index()

print("\n"+"="*70)
print("ASSOCIATION RECOVERY BY CHARACTERISTIC")
print("="*70)
print(results_df.round(3).to_string(index=False))

print("\n"+"="*70)
print("SUMMARY")
print("="*70)
print(summary.round(3).to_string(index=False))


ASSOCIATION RECOVERY BY CHARACTERISTIC
          Model      Characteristic  Empirical_rho  LLM_risk_score_rho  Absolute_difference
claudesonnet4.6                 Age          0.275               0.289                0.014
claudesonnet4.6                 BMI         -0.002               0.209                0.211
claudesonnet4.6 Waist_circumference          0.030               0.251                0.220
claudesonnet4.6         Systolic_BP          0.190               0.350                0.161
claudesonnet4.6        Diastolic_BP          0.029               0.277                0.248
claudesonnet4.6               HbA1c          0.151               0.266                0.115
claudesonnet4.6   Total_cholesterol         -0.192              -0.153                0.038
claudesonnet4.6                 HDL          0.005              -0.270                0.275
claudesonnet4.6                 Sex         -0.115              -0.348                0.233
claudesonnet4.6      Smoking_status     